# Exploratory Data Analysis — Telco Customer Churn

This notebook explores the Telco Customer Churn dataset: distribution, correlations, feature relationships, and customer segmentation.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from utils import load_or_generate_data

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
df = load_or_generate_data(ROOT / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()
df.describe()

## Churn Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

churn_counts = df["Churn"].value_counts()
axes[0].pie(churn_counts, labels=churn_counts.index, autopct="%1.1f%%",
            colors=["#2E86AB", "#E94F37"], startangle=90)
axes[0].set_title("Churn Distribution")

sns.countplot(data=df, x="Churn", hue="Churn", palette={"Yes": "#E94F37", "No": "#2E86AB"}, ax=axes[1], legend=False)
axes[1].set_title("Churn Count")
plt.tight_layout()
plt.show()

## Correlation Heatmap

In [ ]:
num_cols = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
plot_df = df.copy()
plot_df["ChurnBinary"] = (plot_df["Churn"] == "Yes").astype(int)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(plot_df[num_cols + ["ChurnBinary"]].corr(), annot=True, fmt=".2f",
            cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## Customer Segmentation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=df, x="Contract", hue="Churn", palette={"Yes": "#E94F37", "No": "#2E86AB"}, ax=axes[0, 0])
axes[0, 0].set_title("Churn by Contract Type")
axes[0, 0].tick_params(axis="x", rotation=45)

sns.countplot(data=df, x="InternetService", hue="Churn", palette={"Yes": "#E94F37", "No": "#2E86AB"}, ax=axes[0, 1])
axes[0, 1].set_title("Churn by Internet Service")

sns.boxplot(data=df, x="Churn", y="MonthlyCharges", palette={"Yes": "#E94F37", "No": "#2E86AB"}, ax=axes[1, 0])
axes[1, 0].set_title("Monthly Charges by Churn")

sns.boxplot(data=df, x="Churn", y="tenure", palette={"Yes": "#E94F37", "No": "#2E86AB"}, ax=axes[1, 1])
axes[1, 1].set_title("Tenure by Churn")

plt.tight_layout()
plt.show()

## Feature Importance Preview (Random Forest)

In [ ]:
from feature_engineering import FeatureEngineer
from data_preprocessing import DataPreprocessor
from sklearn.ensemble import RandomForestClassifier

engineered = FeatureEngineer().transform(df)
preprocessor = DataPreprocessor()
result = preprocessor.fit_transform(engineered)

rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
rf.fit(result.X_train, result.y_train)

importance = pd.Series(rf.feature_importances_, index=result.feature_names).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
importance.sort_values().plot(kind="barh", ax=ax, color=sns.color_palette("viridis", 15))
ax.set_title("Top 15 Feature Importances (Random Forest)")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()